# Annotation QC — reviewer calibration, triage leak and the edit trail

This notebook measures the **quality-control apparatus** of the Bancada, the annotation
platform in this repository, against the live working database. Every annotation goes
through two independent passes: **triage** (approve or send back — the only pass that
returns work to the annotator) and **Rate and Review** (rate the work as it arrived, fix it
in place with a stated reason for *every* change, rate the result, justify the rating).
A share of the queue is **synthetic**: machine-generated annotations that carry a hidden
target rating and a planted defect family, reviewed blind. They exist to calibrate the
**reviewer** — an instrument almost no annotation platform has, because measuring the
person who checks the work is harder than measuring the person who does it. Everything
below is computed with **the same functions** that `pf annotate export --perfil
quality-report` writes to disk and that the admin dashboard serves over HTTP
(`prompt_factory.annotate.entrega`), so the notebook and the platform cannot drift apart:
there is one implementation of each metric, and this page is a third reader of it.

**Read-only.** Both databases are opened with `readonly=True`. Nothing here writes, and
nothing here runs a pipeline stage.

In [ ]:
# O backend TEM de ser escolhido antes de qualquer figura, e tem de ser o
# `inline`. Ele é headless (renderiza por baixo com o mesmo Agg, então não
# procura display nenhum) E é o único que devolve a figura como SAÍDA DA CÉLULA,
# que é o que o nbconvert embute no HTML. `matplotlib.use("Agg")` puro também
# roda sem display — e faz `plt.show()` virar no-op: o notebook executa limpo,
# sai com código 0 e produz um HTML com ZERO gráfico. Medido aqui: 100% das
# figuras perdidas, sem um aviso sequer.
%matplotlib inline
import base64
import io
import json
from collections import Counter, defaultdict
from html import escape as escape_html

import matplotlib
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# png e não svg: o HTML é commitado, e svg de histograma com dezenas de milhares
# de patches explode o arquivo.
%config InlineBackend.figure_formats = ["png"]

from prompt_factory import paths
from prompt_factory.annotate import db as adb
from prompt_factory.annotate import entrega, geracao
from prompt_factory.db import connect

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "-",
    "figure.facecolor": "white", "axes.facecolor": "white",
})

# Paleta fixa: humano x sintético é a distinção que atravessa o notebook inteiro.
AZUL, LARANJA, CINZA, VERDE, VERMELHO = "#2f6f9f", "#d98032", "#8c8c8c", "#4a8c5f", "#b04a4a"


def mostrar(fig, alt):
    # A figura é embutida À MÃO, com o `alt` escrito por nós. Duas alternativas
    # foram testadas e as duas falham em silêncio:
    #   * `plt.show()` sob `matplotlib.use("Agg")` é NO-OP — o notebook executa
    #     limpo, sai com código 0 e produz um HTML sem um único gráfico;
    #   * `display(fig, metadata={"image/png": {"alt": ...}})` desenha a figura e
    #     DESCARTA o alt: o bloco `data_png` do template `lab` do nbconvert lê
    #     width/height/unconfined/needs_background do metadata e não lê `alt`.
    #     O nbconvert então carimba "No description has been provided for this
    #     image" e avisa no fim da conversão.
    # Emitindo o <img> daqui, o texto alternativo é o que está escrito na chamada
    # — e o comando de render continua sendo UM, sem pós-processamento escondido
    # que quem reproduzir o notebook não teria como adivinhar.
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    plt.close(fig)          # senão o backend inline redesenha a figura no fim da célula
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    display(HTML(f'<img alt="{escape_html(alt, quote=True)}" '
                 f'src="data:image/png;base64,{b64}" '
                 f'style="max-width:100%;height:auto">'))


def anotar_barras(ax, barras, valores, *, dx=0.0, dy=0.0, fmt="{:,.0f}"):
    # Todo gráfico deste notebook carrega o n em cima da barra. Um eixo sozinho
    # esconde a diferença entre "3 de 4" e "3.000 de 4.000", e aqui os n são
    # pequenos o bastante para que essa diferença seja a informação principal.
    for barra, valor in zip(barras, valores, strict=True):
        ax.annotate(fmt.format(valor),
                    (barra.get_width() + dx, barra.get_y() + barra.get_height() / 2 + dy),
                    va="center", ha="left", fontsize=9, color="#333")


# Guard: um clone sem os bancos tem de RENDERIZAR, com a mensagem no lugar do
# número. `--execute` que levanta produz um HTML pela metade, e um HTML pela
# metade é pior que um painel que diz honestamente o que falta.
TEM_BANCADA = paths.ANNOTATE_DB_FILE.exists()
TEM_CORPUS = paths.DB_FILE.exists()
PRONTO = TEM_BANCADA and TEM_CORPUS

print("matplotlib backend:", matplotlib.get_backend())
assert "inline" in matplotlib.get_backend().lower(), (
    "backend sem captura de figura: o HTML sairia sem gráfico nenhum")

conn = conn_corpus = None
if PRONTO:
    conn = connect(paths.ANNOTATE_DB_FILE, readonly=True)
    conn_corpus = connect(paths.DB_FILE, readonly=True)
    print("annotate.sqlite  :", paths.ANNOTATE_DB_FILE)
    print("prompts.sqlite   :", paths.DB_FILE)
else:
    print("MISSING — this notebook renders its structure but not its numbers.")
    print("  annotate.sqlite:", "ok" if TEM_BANCADA else f"absent ({paths.ANNOTATE_DB_FILE})")
    print("  prompts.sqlite :", "ok" if TEM_CORPUS else f"absent ({paths.DB_FILE})")
    print("  build them with:  pf load-db --allow-unlabeled-pct 100   &&   pf annotate seed")

## 0. What exactly was read

An honest fingerprint of the inputs. There is **no wall clock** here on purpose: a
"generated at" timestamp changes the rendered HTML on every run and tells the reader
nothing about the data. What identifies the run is the state of the files.

`db_build_id` is `sha256(universe_sha + labels_sha)[:16]` — the corpus is rebuilt and
swapped underneath the platform by `pf load-db`, so the id is what says *which* corpus the
annotations were made against.

In [ ]:
def _fingerprint(caminho):
    if not caminho.exists():
        return {"file": str(caminho), "state": "absent"}
    st = caminho.stat()
    import datetime
    return {
        "file": str(caminho),
        "bytes": f"{st.st_size:,}",
        # mtime do ARQUIVO, não a hora de agora: é propriedade do insumo.
        "mtime_utc": datetime.datetime.fromtimestamp(
            st.st_mtime, datetime.UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }


if PRONTO:
    fp = {"annotate.sqlite": _fingerprint(paths.ANNOTATE_DB_FILE),
          "prompts.sqlite": _fingerprint(paths.DB_FILE)}
    fp["annotate.sqlite"]["schema_version_anotacao"] = adb.versao_do_banco(conn)
    fp["annotate.sqlite"]["expected_by_this_code"] = adb.SCHEMA_VERSION_ANOTACAO
    meta = {r["key"]: r["value"] for r in conn_corpus.execute("SELECT key, value FROM app_meta")}
    fp["prompts.sqlite"].update({
        "n_rows": f"{int(meta.get('n_rows', 0)):,}",
        "db_build_id": meta.get("db_build_id"),
        "taxonomy_version": meta.get("taxonomy_version"),
        "unlabeled_pct": meta.get("unlabeled_pct"),
    })
    pool = {r["key"]: r["value"] for r in conn.execute(
        "SELECT key, value FROM app_meta WHERE key LIKE 'pool_%'")}
    fp["prompt pool"] = {
        "origin": pool.get("pool_origem"),
        "corpus_build_it_was_built_from": pool.get("pool_db_build_id"),
        "rows_excluded_by_license_policy": pool.get("pool_excluidas_licenca"),
    }
    print(json.dumps(fp, indent=2, ensure_ascii=False))
    if fp["prompt pool"]["corpus_build_it_was_built_from"] != meta.get("db_build_id"):
        print("\nNOTE: the materialised pool was built from a DIFFERENT corpus build "
              "than the one on disk. The platform rebuilds it when the signature changes.")
else:
    print("skipped — no database")

## 1. Two different questions: what would ship, and what exists

These are not the same set, and conflating them is the easiest way to make a QC report
flatter itself.

* **`entrega.coletar(...)`** answers *"what would ship right now?"*. It admits only
  `status = 'avaliada'` — the single terminal state of acceptance in `db.TRANSICOES`, i.e.
  the item cleared **both** passes. With `incluir_pendentes=True` it also admits
  `pendente_avaliacao` (cleared triage, second pass not done); with
  `incluir_sinteticas=True` the synthetic items are not filtered out. Both flags are set
  below because that is exactly what the admin dashboard uses — the defaults are the
  *delivery* defaults, and this notebook is describing the **workshop**.
* **The operational ledger** — plain `SELECT`s below — answers *"what is in the platform,
  in whatever state?"*. Work sent back by triage, work waiting to be triaged, and
  synthetic items still queued all live here and in none of the delivery profiles.

The gap between the two is the honest picture of a platform mid-flight.

In [ ]:
if PRONTO:
    # Os dois True: mesmo conjunto que /api/admin/metricas monta.
    itens = entrega.coletar(conn, conn_corpus,
                            incluir_sinteticas=True, incluir_pendentes=True)
    print(f"entrega.coletar(incluir_sinteticas=True, incluir_pendentes=True) "
          f"-> {len(itens)} annotation(s) in a deliverable state")
    for i in itens:
        print(f"    #{i['anotacao_id']}  {i['tipo']:<16} {i['status']:<20} "
              f"synthetic={i['sintetica']!s:<5} corrected={i['corrigido']}")

    # ------------------------------------------------------------------ ledger
    # "operational ledger view": SELECT próprio, fora de `entrega`, porque
    # `entrega` só enxerga o que é entregável — que é a função dele.
    ledger = defaultdict(lambda: {"human": 0, "synthetic": 0})
    for r in conn.execute(
        "SELECT status, (gabarito_avaliacao_json IS NOT NULL) AS sint, count(*) AS n "
        "FROM anotacoes GROUP BY 1, 2"
    ):
        ledger[r["status"]]["synthetic" if r["sint"] else "human"] += r["n"]
    total_ledger = sum(v["human"] + v["synthetic"] for v in ledger.values())
    entregaveis = len(itens)

    print(f"\noperational ledger: {total_ledger} annotation(s) in the platform, all states")
    print(f"  of which deliverable today: {entregaveis} "
          f"({entregaveis / total_ledger:.0%} of {total_ledger})")

    ordem = [s for s in adb.STATUS_ANOTACAO if s in ledger]
    ordem += [s for s in ledger if s not in ordem]          # status desconhecido nunca some
    humanos = [ledger[s]["human"] for s in ordem]
    sints = [ledger[s]["synthetic"] for s in ordem]

    fig, ax = plt.subplots(figsize=(9, 0.62 * len(ordem) + 1.9))
    b1 = ax.barh(ordem, humanos, color=AZUL, label="human")
    b2 = ax.barh(ordem, sints, left=humanos, color=LARANJA, label="synthetic")
    for y, (h, s) in enumerate(zip(humanos, sints, strict=True)):
        ax.annotate(f"{h + s}", (h + s + 0.12, y), va="center", fontsize=9, color="#333")
        if s:
            ax.annotate(f"{s} synth", (h + s / 2, y), va="center", ha="center",
                        fontsize=8, color="white", weight="bold")
    ax.set_xlim(0, max(h + s for h, s in zip(humanos, sints, strict=True)) * 1.22)
    ax.invert_yaxis()
    ax.set_xlabel("annotations")
    ax.set_title(f"Annotations by status — operational ledger (n = {total_ledger})")
    ax.legend(loc="lower right", frameon=False)
    ax.annotate(
        f"deliverable ('avaliada'): {entregaveis} of {total_ledger}",
        xy=(0.99, 1.02), xycoords="axes fraction", ha="right", fontsize=9, color="#555")
    plt.tight_layout()
    mostrar(fig, "Horizontal stacked bar of annotations by workflow status, split into human and synthetic segments, with the count printed on each bar.")

    faltando = [s for s in adb.STATUS_ANOTACAO if s not in ledger]
    if faltando:
        print("states with zero rows today: " + ", ".join(faltando))
else:
    itens, ledger, total_ledger = [], {}, 0
    print("skipped — no database")

## 2. Human and synthetic, declared rather than inferred

The signal that an annotation is machine-generated is **one column**, and the singleness is
the decision: `anotacoes.gabarito_avaliacao_json IS NOT NULL`. Two marks would eventually
disagree, and on the day they disagreed nobody would know which to believe. Synthetic
annotations stay out of the data exports by default — the project does not deliver machine
annotation as human work — and stay *in* the QC artifacts, because measuring the reviewer
is what they are for.

> **Withheld on purpose.** `geracao.composicao()` also returns the target rating and the
> planted defect family of every synthetic item. Those breakdowns are printed below **only
> for synthetic items that have already been reviewed**. The ones still in the queue are
> the reviewer's next job, and this HTML is committed to the repository: publishing the
> multiset of pending targets would let a reviewer allocate ratings by elimination, and
> the calibration number further down would stop measuring anything.

In [ ]:
if PRONTO:
    comp = geracao.composicao(conn)

    # Quais sintéticas JÁ foram avaliadas? Só sobre elas um agregado por
    # alvo/família pode ser publicado.
    sint_avaliadas = {r["anotacao_id"] for r in conn.execute(
        "SELECT av.anotacao_id FROM avaliacoes av "
        "JOIN anotacoes an ON an.id = av.anotacao_id "
        "WHERE an.gabarito_avaliacao_json IS NOT NULL")}
    n_sint = int(comp.get("sinteticas") or 0)
    n_pendentes = n_sint - len(sint_avaliadas)

    seguro = {k: v for k, v in comp.items()
              if k not in ("sinteticas_por_alvo", "sinteticas_por_familia")}
    print(json.dumps(seguro, indent=2, ensure_ascii=False))

    print(f"\nsynthetic items reviewed so far : {len(sint_avaliadas)} of {n_sint}")
    print(f"synthetic items still queued    : {n_pendentes}")
    if n_pendentes:
        print("\n  -> target-rating and defect-family breakdowns are WITHHELD from this")
        print("     notebook while any synthetic item is still awaiting review.")
        print("     They are in the database (and in `geracao.composicao`) — they are not")
        print("     printed into a committed HTML page that the reviewer can read.")
    else:
        print("\ntarget / defect-family distribution (all synthetic items already reviewed):")
        print(json.dumps({k: comp.get(k) for k in
                          ("sinteticas_por_alvo", "sinteticas_por_familia")},
                         indent=2, ensure_ascii=False))

    fig, ax = plt.subplots(figsize=(7.2, 2.0))
    h, s = int(comp.get("humanas") or 0), n_sint
    b1 = ax.barh(["annotations"], [h], color=AZUL, label="human")
    b2 = ax.barh(["annotations"], [s], left=[h], color=LARANJA, label="synthetic")
    ax.annotate(f"{h} human", (h / 2, 0), va="center", ha="center",
                color="white", weight="bold", fontsize=9)
    if s:
        ax.annotate(f"{s} synthetic", (h + s / 2, 0), va="center", ha="center",
                    color="white", weight="bold", fontsize=9)
    ax.set_xlim(0, h + s)
    ax.set_yticks([])
    ax.grid(False)
    ax.set_xlabel("annotations in the platform, all states")
    ax.set_title(f"Composition (n = {h + s}) — signal: {comp.get('sinal')}", fontsize=9.5)
    ax.legend(loc="lower right", frameon=False, fontsize=8)
    plt.tight_layout()
    mostrar(fig, "Single horizontal bar splitting all annotations in the platform into a human segment and a synthetic segment, each labelled with its count.")

    mat = comp.get("material_por_origem") or {}
    if mat:
        print("Material the tasks stand on, by provenance "
              "('importada' = produced by the P4c generation campaign):")
        for tabela, origens in mat.items():
            linha = ", ".join(f"{k}={v}" for k, v in sorted(origens.items()))
            print(f"    {tabela:<18} {linha}")
else:
    comp, sint_avaliadas, n_pendentes = {}, set(), 0
    print("skipped — no database")

## 3. Reviewer calibration against the hidden targets

Synthetic annotations carry a target rating written **before** the review and never visible
in the interface, in the API envelope or in the page source. Comparing the reviewer's
`avaliacao_antes` to that target is the calibration measurement.

The scale is **ordinal** (`inutilizavel < ajustavel < adequado < excepcional`), so distance
matters: calling `adequado` what was `excepcional` is a one-step miss; calling `adequado`
what was `inutilizavel` is three. `entrega.calibracao_do_revisor` therefore returns exact
agreement, within-one agreement **and** mean absolute error — an exact-match rate alone
would flatten those two into the same "wrong".

When no synthetic item has been reviewed yet, the function returns `{"n": 0, "scale": ...}`
and **nothing else** — no `exact_rate` key at all. That absence is deliberate and is the
same lesson as the labeling campaign's agreement: **`None` and `0.0` are different
statements**, and reporting a rate of zero for "not measured" would read as a reviewer who
gets everything wrong.

In [ ]:
if PRONTO:
    cal = entrega.calibracao_do_revisor(itens)
    print(json.dumps(cal, indent=2, ensure_ascii=False))

    if not cal.get("n"):
        # Painel "not measured yet". Mostra a CONTAGEM e o caminho que destrava —
        # nunca o alvo ou a família de um item pendente.
        fig, ax = plt.subplots(figsize=(9, 2.7))
        ax.axis("off")
        escala = " < ".join(cal.get("scale", adb.AVALIACOES_ANTES))
        ax.text(0.5, 0.88, "Reviewer calibration: NOT MEASURED YET",
                ha="center", fontsize=13, weight="bold", color=CINZA)
        ax.text(0.5, 0.62,
                f"{len(sint_avaliadas)} of {int(comp.get('sinteticas') or 0)} synthetic "
                f"annotations have been through Rate and Review.",
                ha="center", fontsize=10.5)
        ax.text(0.5, 0.44, f"ordinal scale in use:  {escala}",
                ha="center", fontsize=9.5, color="#555", family="monospace")
        ax.text(0.5, 0.20,
                "This is not a score of 0. It is the absence of a measurement — the\n"
                "distinction the metric is built to preserve.",
                ha="center", fontsize=9.5, color=CINZA, style="italic")
        for s in ax.spines.values():
            s.set_visible(False)
        ax.add_patch(plt.Rectangle((0.02, 0.05), 0.96, 0.9, fill=False,
                                   edgecolor=CINZA, linewidth=1.1, linestyle="--",
                                   transform=ax.transAxes))
        plt.tight_layout()
        mostrar(fig, "Framed text panel stating that reviewer calibration has not been measured yet, giving how many synthetic items have been reviewed and the ordinal scale in use.")

        print("What unlocks this panel — and only the owner can do it, which is the point:")
        print("  1. open the Bancada:            pf annotate                (http://127.0.0.1:8766)")
        print("  2. triage the queued synthetic items, blind, as any other item")
        print("  3. rate them in Rate and Review — the hidden target is revealed to")
        print("     nobody until the evaluation row exists")
        print("  4. re-run this notebook; the confusion matrix below replaces this panel")
    else:
        escala = cal["scale"]
        idx = {n: i for i, n in enumerate(escala)}
        M = [[0] * len(escala) for _ in escala]
        for chave, n in cal["confusion"].items():
            esperado, obtido = [p.strip() for p in chave.split("->")]
            M[idx[esperado]][idx[obtido]] = n

        fig, ax = plt.subplots(figsize=(6.4, 5.4))
        im = ax.imshow(M, cmap="Blues", vmin=0)
        ax.set_xticks(range(len(escala)), escala, rotation=30, ha="right")
        ax.set_yticks(range(len(escala)), escala)
        ax.set_xlabel("reviewer said")
        ax.set_ylabel("hidden target")
        ax.set_title(f"Reviewer vs hidden target (n = {cal['n']})")
        ax.grid(False)
        limite = max(max(linha) for linha in M) or 1
        for a in range(len(escala)):
            for b in range(len(escala)):
                if M[a][b]:
                    ax.text(b, a, M[a][b], ha="center", va="center", fontsize=11,
                            color="white" if M[a][b] > limite * 0.6 else "#222",
                            weight="bold")
        for a in range(len(escala)):      # a diagonal é o acerto exato
            ax.add_patch(plt.Rectangle((a - 0.5, a - 0.5), 1, 1, fill=False,
                                       edgecolor=VERDE, linewidth=2))
        fig.colorbar(im, ax=ax, shrink=0.75, label="annotations")
        plt.tight_layout()
        mostrar(fig, "Two side-by-side horizontal bar charts: evaluations rated on arrival, and rated after correction, each drawn over the full ordinal scale including empty levels.")

        print(f"exact agreement : {cal['exact']}/{cal['n']} ({cal['exact_rate']:.1%})")
        print(f"within one step : {cal['within_one']}/{cal['n']} ({cal['within_one_rate']:.1%})")
        print(f"mean absolute error: {cal['mean_absolute_error']} step(s) on a "
              f"{len(escala)}-point ordinal scale")
        print("\nBy planted defect family (n = how many carried it, exact = how many the "
              "reviewer rated on target):")
        for fam, v in (cal.get("by_defect_family") or {}).items():
            print(f"    {fam:<34} n={v['n']:<4} exact={v['exact']}")
else:
    cal = {}
    print("skipped — no database")

## 4. Did triage let anything through?

This metric measures **the triage reviewer**, not the annotator, which is why it exists
under its own name instead of being diluted into a general rejection rate. An item approved
in pass 1 and found `inutilizavel` in pass 2 went through a gate that should have stopped
it — and the interesting case is not the one that gets discarded. It is the one that gets
**rescued**: rated unusable on arrival, fixed in place, and delivered anyway. That item
ships, so a metric that counted only discards would hide the most common way triage fails,
namely the one somebody paid to repair.

Reported at two scopes, because they answer different questions and today they differ.

In [ ]:
if PRONTO:
    vaz = entrega.triagem_deixou_passar(itens)
    print("delivery scope (the numbers `pf annotate export --perfil quality-report` writes):")
    print(json.dumps(vaz, indent=2, ensure_ascii=False))

    # Escopo LEDGER: toda avaliação já feita, inclusive sobre item que a
    # escalação devolveu depois e que hoje não é entregável.
    linhas = conn.execute(
        "SELECT av.avaliacao_antes AS antes, av.avaliacao_depois AS depois, "
        "       an.status, (an.gabarito_avaliacao_json IS NOT NULL) AS sint "
        "FROM avaliacoes av JOIN anotacoes an ON an.id = av.anotacao_id"
    ).fetchall()
    n_av = len(linhas)
    chegaram_ruins = sum(1 for r in linhas if r["antes"] == "inutilizavel")
    terminaram_ruins = sum(1 for r in linhas if r["depois"] == "incorrigivel")
    resgatados = sum(1 for r in linhas
                     if r["antes"] == "inutilizavel" and r["depois"] in ("adequado", "excepcional"))

    print(f"\noperational ledger scope: {n_av} evaluation(s) recorded in total")
    if n_av:
        print(f"  arrived 'inutilizavel'   : {chegaram_ruins} of {n_av} "
              f"({chegaram_ruins / n_av:.0%})   <- triage leak")
        print("  of those, rescued        : "
              + (f"{resgatados} of {chegaram_ruins}" if chegaram_ruins
                 else "n/a - nothing leaked through triage, so there was nothing to rescue"))
        print(f"  ended 'incorrigivel'     : {terminaram_ruins} of {n_av} "
              f"({terminaram_ruins / n_av:.0%})   <- paid for twice, discarded")

        antes = Counter(r["antes"] for r in linhas)
        depois = Counter(r["depois"] for r in linhas)
        fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2), sharex=True)
        for ax, cont, escala, titulo, cor in (
            (axes[0], antes, adb.AVALIACOES_ANTES, "Rated on arrival", AZUL),
            (axes[1], depois, adb.AVALIACOES_DEPOIS, "Rated after correction", VERDE),
        ):
            rot = list(escala)
            val = [cont.get(k, 0) for k in rot]
            barras = ax.barh(rot, val, color=cor)
            anotar_barras(ax, barras, val, dx=max(val) * 0.03 if max(val) else 0.05)
            ax.invert_yaxis()
            ax.set_xlim(0, (max(val) or 1) * 1.28)
            ax.set_title(f"{titulo} (n = {n_av})")
            ax.set_xlabel("evaluations")
        plt.tight_layout()
        mostrar(fig, "Horizontal bar chart counting second-pass edits grouped by the top-level payload field they touched.")
        print("Both panels are the whole ordinal scale, zeros included: a scale drawn only "
              "where it has mass hides which outcomes were available and never happened.")
    else:
        print("  no evaluation recorded yet — the second pass has not run.")
else:
    vaz, n_av = {}, 0
    print("skipped — no database")

## 5. The edit trail: every change with the reason given for it

The second pass does not send work back — it **corrects in place**. What makes that
auditable rather than arbitrary is that the server, not the client, computes the diff
between the submitted payload and the corrected one, flattens it to leaf paths
(`{"notas": [{"nota": 3}]}` → `notas.0.nota`) and requires **exact** correspondence between
the paths that changed and the reasons declared. Both directions are refused: a change
without a reason is somebody silently altering another person's work; a reason for a field
that did not change makes the audit describe a correction that never happened, which is
worse than having no audit.

Every row below is one field, one before, one after, one stated reason.

In [ ]:
if PRONTO:
    edicoes = conn.execute(
        "SELECT e.campo, e.valor_antes, e.valor_depois, e.motivo, "
        "       av.anotacao_id, av.avaliacao_antes, av.avaliacao_depois "
        "FROM edicoes_avaliacao e JOIN avaliacoes av ON av.id = e.avaliacao_id "
        "ORDER BY av.anotacao_id, e.id"
    ).fetchall()
    n_ed = len(edicoes)
    print(f"{n_ed} individual field edit(s) across {n_av} evaluation(s)"
          + (f" — {n_ed / n_av:.1f} per evaluation" if n_av else ""))

    def _mudanca(antes, depois):
        # Truncar os dois lados no mesmo ponto é inútil quando a mudança está
        # depois do corte: sai "from" e "to" IDÊNTICOS e a auditoria parece
        # descrever uma correção que não houve — exatamente o que a validação do
        # diff existe para impedir. Em texto longo, mostra ONDE divergiu.
        a = "" if antes is None else str(antes)
        d = "" if depois is None else str(depois)
        if max(len(a), len(d)) <= 100:
            return [f"from  : {a or '(empty)'}", f"to    : {d or '(empty)'}"]
        i = 0
        while i < min(len(a), len(d)) and a[i] == d[i]:
            i += 1
        return [
            f"from  : {len(a)} chars", f"to    : {len(d)} chars",
            f"first divergence at char {i} of the common prefix:",
            f"    was : ...{a[i:i + 60]}",
            f"    now : ...{d[i:i + 60]}",
        ]

    if n_ed:
        for r in edicoes:
            print(f"\n  annotation #{r['anotacao_id']}  field `{r['campo']}`")
            for linha in _mudanca(r["valor_antes"], r["valor_depois"]):
                print(f"      {linha}")
            print(f"      reason: {r['motivo']}")

        # Prefixo = a chave de topo do caminho achatado. É o que diz QUE PARTE do
        # instrumento o revisor mexe — nota, justificativa, âncora da rubrica.
        prefixos = Counter(str(r["campo"]).split(".")[0] for r in edicoes)
        rot = [k for k, _ in prefixos.most_common()]
        val = [prefixos[k] for k in rot]
        fig, ax = plt.subplots(figsize=(8, 0.55 * len(rot) + 1.7))
        barras = ax.barh(rot, val, color=LARANJA)
        anotar_barras(ax, barras, val, dx=max(val) * 0.03)
        ax.invert_yaxis()
        ax.set_xlim(0, max(val) * 1.25)
        ax.set_xlabel("edits")
        ax.set_ylabel("payload field (top-level path)")
        ax.set_title(f"What the second pass actually corrects (n = {n_ed} edits)")
        plt.tight_layout()
        mostrar(fig, "Strip plot of active minutes per annotation, one row per task type, one dot per annotation, with a red tick marking each row's median.")

        anot_tocadas = len({r["anotacao_id"] for r in edicoes})
        print(f"{anot_tocadas} annotation(s) were corrected in place; "
              f"{n_av - anot_tocadas} passed through unchanged.")
    else:
        print("no edit recorded — either nothing needed fixing, or pass 2 has not run.")
else:
    print("skipped — no database")

## 6. Active time per task type

`tempo_ativo_ms` is time the tab was actually in front of a person, not wall-clock from
claim to submit. Synthetic annotations are written by a batch importer and carry **0 ms**;
they are excluded from this chart and counted separately, because averaging a machine's
zero into a human median is the cheapest way to make a platform look fast.

With this few points the honest display is the **points themselves**, not a box plot: a box
plot over four observations draws quartiles that are an artifact of having four
observations.

In [ ]:
if PRONTO:
    linhas = conn.execute(
        "SELECT t.tipo, an.tempo_ativo_ms AS ms, "
        "       (an.gabarito_avaliacao_json IS NOT NULL) AS sint "
        "FROM anotacoes an "
        "JOIN atribuicoes a ON a.id = an.atribuicao_id "
        "JOIN tarefas t     ON t.id = a.tarefa_id"
    ).fetchall()
    humanos = [r for r in linhas if not r["sint"]]
    sinteticas = [r for r in linhas if r["sint"]]
    zerados = [r for r in humanos if not r["ms"]]
    validos = [r for r in humanos if r["ms"]]

    print(f"{len(linhas)} annotation(s): {len(humanos)} human, {len(sinteticas)} synthetic")
    print(f"synthetic excluded from the chart (all {len(sinteticas)} carry 0 ms, "
          f"written by the batch importer)")
    if zerados:
        print(f"{len(zerados)} human annotation(s) also carry 0 ms and are excluded")

    if validos:
        por_tipo = defaultdict(list)
        for r in validos:
            por_tipo[str(r["tipo"])].append(int(r["ms"]) / 60000.0)
        tipos = [t for t in adb.TIPOS_TAREFA if t in por_tipo]
        tipos += [t for t in por_tipo if t not in tipos]

        fig, ax = plt.subplots(figsize=(9, 0.56 * len(tipos) + 2.0))
        for y, t in enumerate(tipos):
            vs = sorted(por_tipo[t])
            ax.scatter(vs, [y] * len(vs), s=64, color=AZUL, zorder=3,
                       alpha=0.85, edgecolor="white", linewidth=1.1)
            mediana = vs[len(vs) // 2]
            ax.scatter([mediana], [y], s=150, marker="|", color=VERMELHO, zorder=4)
            ax.annotate(f"n={len(vs)}", (ax.get_xlim()[0], y), xytext=(-6, 0),
                        textcoords="offset points", ha="right", va="center",
                        fontsize=8.5, color="#555")
        ax.set_yticks(range(len(tipos)), tipos)
        ax.invert_yaxis()
        ax.set_xlabel("active time (minutes)")
        ax.set_title(f"Active time per annotation, human only (n = {len(validos)})")
        ax.set_xlim(left=0)
        ax.annotate("red tick = median", xy=(0.99, 1.03), xycoords="axes fraction",
                    ha="right", fontsize=8.5, color=VERMELHO)
        plt.tight_layout()
        mostrar(fig, "Confusion matrix heatmap of the reviewer's rating against the hidden target rating, with the exact-agreement diagonal outlined in green.")

        todos = sorted(int(r["ms"]) for r in validos)
        print(f"\nmedian across all human annotations: "
              f"{todos[len(todos) // 2] / 60000:.1f} min "
              f"(range {todos[0] / 60000:.1f} to {todos[-1] / 60000:.1f} min, "
              f"n = {len(todos)})")
        print("These are minutes of real work per item. They are a plausibility check on "
              "the data, not a productivity target: n is small and one person produced most of it.")
    else:
        print("no human annotation with recorded active time yet.")
else:
    print("skipped — no database")

## 7. One item, end to end

The strongest single artifact this platform produces is the **audit** of one item: the
prompt with its licence, the annotation as submitted, the triage verdict, every second-pass
edit with the reason given for it, the ratings before and after, and the final decision.

The example below is the first item of the deliverable set — whichever that is on the day
this page was rendered, rather than a specimen chosen to look good. Its provenance block
prints whatever the prompt actually carries: a dataset name and a licence when it came from
the corpus, or `source = demonstration pack` with `is_demo = true` when it came from the
hand-written pack. Those two axes are independent and the platform keeps them apart:
`is_demo` says who wrote the **prompt**, `synthetic` says who wrote the **annotation**. A
demo prompt annotated by a person is delivered *marked*, not dropped — confusing the two
would discard real human work for having been done on a fixture.

> **Prompt text is deliberately not reproduced here.** This notebook renders to an HTML
> page committed to the repository, and the rule it follows is uniform: no prompt body from
> the corpus, the seed or the demonstration pack is printed into it. What is shown is the
> provenance block, the *shape* of the payload and the QC chain. The full text lives in
> `pf annotate export --perfil audit --anotacao <id>`, which is a local artifact.

In [ ]:
if PRONTO and itens:
    it = itens[0]
    proc = entrega._licenca(it)      # a MESMA função que carimba a licença no export
    print(f"=== annotation #{it['anotacao_id']} — {it['tipo']} "
          f"(contract {it['payload_schema']}) ===\n")

    print("1. PROVENANCE OF THE PROMPT")
    for k in ("uid", "lang", "source", "license", "license_class", "attribution", "is_demo"):
        print(f"      {k:<14}: {proc.get(k)}")
    print(f"      {'text':<14}: [{len(str((it['prompt'] or {}).get('text') or ''))} chars — "
          f"not reproduced in this notebook]")

    def _escalar(x):
        # Número e booleano saem inteiros (são a NOTA, o dado); texto vira só o
        # tamanho. É o que prova o contrato sem publicar prosa de ninguém.
        if isinstance(x, bool) or x is None or isinstance(x, (int, float)):
            return str(x)
        return f"<text, {len(str(x))} chars>"

    def _forma(v, prof=0):
        pad = "  " * prof
        out = []
        if isinstance(v, dict):
            for k, x in v.items():
                if isinstance(x, (dict, list)):
                    out.append(f"{pad}{k}:")
                    out += _forma(x, prof + 1)
                else:
                    out.append(f"{pad}{k}: {_escalar(x)}")
        elif isinstance(v, list):
            out.append(f"{pad}[{len(v)} item(s)]")
            for i, x in enumerate(v):
                if isinstance(x, (dict, list)):
                    out.append(f"{pad}  [{i}]")
                    out += _forma(x, prof + 2)
                else:
                    out.append(f"{pad}  [{i}] {_escalar(x)}")
        else:
            out.append(f"{pad}{_escalar(v)}")
        return out

    print("\n2. THE ANNOTATION — payload shape as submitted")
    print("      guideline v" + str(it["versao_diretriz"]) +
          (f" · project brief v{it['versao_brief']}" if it["versao_brief"] else
           " · project brief: none (predates the brief)"))
    for linha in _forma(it["payload_submetido"]):
        print("      " + linha)

    print("\n3. TRIAGE (pass 1)")
    tri = it["triagem"]
    if tri:
        print(f"      verdict : {tri['verdict']} by {tri['reviewer']}"
              + (" (self-review)" if tri["self_review"] else ""))
        print(f"      comment : {tri['comment'] or '(none)'}")
    else:
        print("      not triaged")

    print("\n4. RATE AND REVIEW (pass 2)")
    av = it["avaliacao"]
    if av:
        print(f"      as it arrived    : {av['avaliacao_antes']}")
        print(f"      after correction : {av['avaliacao_depois']}")
        print(f"      reviewer         : {av['revisor']}"
              + (" (self-review)" if av["autorrevisao"] else ""))
        print(f"      rationale        : {av['justificativa']}")
        print(f"      edits            : {len(av['edicoes'])}")
        for e in av["edicoes"]:
            print(f"          `{e['campo']}`  {str(e['valor_antes'])[:40]!r} -> "
                  f"{str(e['valor_depois'])[:40]!r}")
            print(f"             because: {e['motivo']}")
    else:
        print("      not rated yet")

    print(f"\n5. OUTCOME — final status: {it['status']}"
          + ("  (escalated: " + it["decisao_admin"]["decision"] + ")"
             if it["decisao_admin"] else ""))
    print(f"   synthetic: {it['sintetica']}  ·  corrected by reviewer: {it['corrigido']}"
          f"  ·  active time: {it['tempo_ativo_ms'] / 1000:.1f}s")
elif PRONTO:
    print("no annotation in a deliverable state yet — nothing to walk through.")
else:
    print("skipped — no database")

## 8. What this cannot claim yet

Stating the limits is part of the instrument; a QC report that does not know its own n is
not a QC report.

* **n is tiny.** The whole ledger is a double-digit number of annotations. Every rate above
  is a *pipeline proof* — the metric is wired end to end and computes from real rows — and
  **not statistical evidence** about annotator or reviewer quality. No confidence interval
  would be honest at this n, so none is drawn.
* **Reviewer calibration is not measured.** Synthetic items are queued and unreviewed. The
  panel says "not measured", never `0.0`, and it withholds their targets so that the
  measurement remains possible.
* **The triage-leak rate rests on a handful of evaluations.** A leak rate of 0% over that
  many items means "no leak has been observed yet", which is a much weaker statement than
  "triage does not leak".
* **One operator, and the data says so.** This is a portfolio platform run largely by one
  person. Where the same human both annotated and reviewed, the row carries
  `autorrevisao = 1` — recorded at write time, not derived later, because
  `atribuicoes.anotador_id` can be corrected and a derivation would then quietly start
  lying. Self-review is a declared property of the data here, not a hidden one.
* **Active time is a plausibility check, not a benchmark.** Minutes per item are consistent
  with real work; they are not a throughput claim.
* **Nothing here is broken down by prompt category**, and the reason is not that the corpus
  lacks labels — it has them now, on two of the four axes. `task_type` and `domain` are
  populated for most of the universe, mostly by a classifier with per-row abstention;
  `quality` was trained, measured and deliberately **not** applied, and `nsfw` had too few
  positives to train at all. The obstacle here is n: cutting a few dozen annotations by
  sixteen task types would produce cells of zero and one. The second notebook covers the
  campaign that produced those labels, and what that campaign can and cannot claim.

## Closing

Every number above came from `prompt_factory.annotate.entrega` and
`prompt_factory.annotate.geracao` — the same modules that write
`pf annotate export --perfil quality-report` to disk and that the admin dashboard serves
over HTTP. There is one implementation of reviewer calibration, one of the triage-leak
metric, one definition of what counts as delivered. **The notebook and the platform cannot
diverge**, because there is nothing here for them to diverge *about*: this page is a third
reader of the same functions, not a second computation of the same idea.

---

### Licence and provenance of this page

The prompts underlying this work are ingested from public conversation corpora under
mixed licences (ODC-BY-1.0, CC-BY-4.0, CC-BY-SA-3.0, CC-BY-NC-4.0, Apache-2.0, MIT), and
the platform's prompt pool already excludes rows that are not redistributable. This
rendered page contains **counts, distributions and QC metadata only** — no prompt body from
the corpus, the seed or the demonstration pack, and no target rating of any synthetic item
still awaiting review. Attribution for any actual data export is emitted per row by
`prompt_factory.export`, resolved from `config/sources.toml`.

In [ ]:
# Fechar as conexões. Conexão SOMENTE-LEITURA em WAL cria o `-shm` ao lado do
# banco e NÃO consegue removê-lo ao fechar (remover é escrita), e um `-shm`
# órfão é exatamente o sinal que o pré-voo do swap do `pf load-db` lê como
# "alguém está com este banco aberto" — a próxima recarga seria recusada por
# causa de um kernel já morto. Rode `pf annotate status` depois deste notebook.
for c in (conn, conn_corpus):
    if c is not None:
        c.close()
print("connections closed.")
for sufixo in ("-shm", "-wal"):
    for base in (paths.DB_FILE, paths.ANNOTATE_DB_FILE):
        p = base.with_name(base.name + sufixo)
        if p.exists():
            print(f"  left behind: {p.name} ({p.stat().st_size:,} bytes)"
                  + ("   <- run `pf annotate status` to clear it" if sufixo == "-shm" else ""))